# MIMO Multi-Output: Baseline vs Feature Engineering

So sánh 2 phiên bản train trên cùng 1 pipeline MIMO (t+1 → t+6):

| | Baseline | FE version |
|---|---|---|
| Input features | Các biến gốc từ `clean_data.csv` | Lags, rolling, cyclical + meteo gốc |
| Target | `pm25_avg` shift -1 đến -6 | Như nhau |
| NaN handling | `dropna()` per segment (không tạo lag cross-gap) | Như nhau |
| Models | LR, KNN, RF, XGB, CatBoost | Như nhau |

**Lý do giữ NaN trong `clean_data.csv`:**  
Dataset có large gaps (nhiều giờ liên tiếp thiếu dữ liệu).  
Nếu forward-fill hoặc interpolate trước khi tạo lag, `lag_1h` tại điểm đầu mỗi segment  
sẽ lấy giá trị từ segment trước → lag sai về mặt vật lý.  
Giải pháp: tạo lag trên series gốc (có NaN), sau đó `dropna()` — hàng đầu mỗi segment  
tự động bị loại vì lag của nó là NaN.

## 0. Setup

In [12]:
# pip install scikit-learn 

In [13]:
# pip install xgboost

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

HORIZON = 6
TARGET_COLS = [f'target_h{h}' for h in range(1, HORIZON + 1)]
RANDOM_STATE = 42

## 1. Load data

In [17]:
import os
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent 
PROCESSED_FILE = PROJECT_ROOT / "data" / "processed" / "clean_data.csv"

df_raw = pd.read_csv(
    PROCESSED_FILE,
    parse_dates=['datetime'],
    index_col='datetime'
)
# Chuẩn hóa timezone về UTC-naive để tránh lỗi khi tính toán
df_raw.index = df_raw.index.tz_localize(None) if df_raw.index.tz is None \
               else df_raw.index.tz_convert(None)
df_raw = df_raw.sort_index()

print('Shape:', df_raw.shape)
print('Columns:', df_raw.columns.tolist())
print(f'\nTime range: {df_raw.index.min()} → {df_raw.index.max()}')
print(f'NaN in pm25_avg: {df_raw["pm25_avg"].isna().sum()} rows')
print(f'NaN total: {df_raw.isna().sum().sum()} cells')
display(df_raw.head())

Shape: (11726, 10)
Columns: ['pm25_avg', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'wind_direction_10m', 'surface_pressure', 'boundary_layer_height', 'wind_u', 'wind_v']

Time range: 2024-11-19 10:00:00 → 2026-03-22 23:00:00
NaN in pm25_avg: 1077 rows
NaN total: 1077 cells


,pm25_avg,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,surface_pressure,boundary_layer_height,wind_u,wind_v
datetime,,,,,,,,,,
2024-11-19 10:00:00,29.139999,32.0,55,0.0,1.57,59,1006.8,910.0,-1.345753,-0.808610
2024-11-19 11:00:00,29.150000,30.9,59,0.1,0.29,211,1007.6,150.0,0.149361,0.248579
2024-11-19 12:00:00,31.783333,29.3,68,0.0,1.37,204,1008.4,245.0,0.557229,1.251557
2024-11-19 13:00:00,30.950000,26.6,77,0.2,0.78,105,1009.5,95.0,-0.753422,0.201879
2024-11-19 14:00:00,30.216667,26.6,79,0.1,1.10,309,1010.0,85.0,0.854861,-0.692252


## 2. Hàm tiện ích chung

In [18]:
def make_targets(df, horizon=6):
    """Tạo 6 cột target bằng shift âm trên pm25_avg.
    NaN ở cuối mỗi gap sẽ được loại tự động khi dropna().
    """
    out = df.copy()
    for h in range(1, horizon + 1):
        out[f'target_h{h}'] = out['pm25_avg'].shift(-h)
    return out


def chrono_split(df, train_ratio=0.70, valid_ratio=0.15):
    """Split chronological — không shuffle."""
    n = len(df)
    train_end = int(n * train_ratio)
    valid_end = int(n * (train_ratio + valid_ratio))
    return (
        df.iloc[:train_end],
        df.iloc[train_end:valid_end],
        df.iloc[valid_end:]
    )


def build_preprocessor(X, categorical_cols=None):
    """ColumnTransformer: OneHot cho cat, StandardScaler cho numeric."""
    categorical_cols = categorical_cols or []
    numeric_cols = [c for c in X.columns if c not in categorical_cols]
    transformers = [('num', StandardScaler(), numeric_cols)]
    if categorical_cols:
        transformers.insert(0, ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols))
    return ColumnTransformer(transformers=transformers)


def evaluate_mimo(Y_true, Y_pred, horizon=6):
    """Tính MAE, RMSE, R2 per horizon + overall average."""
    rows = []
    for h in range(horizon):
        yt, yp = Y_true[:, h], Y_pred[:, h]
        rows.append({
            'Horizon': f't+{h+1}h',
            'MAE':  round(mean_absolute_error(yt, yp), 3),
            'RMSE': round(np.sqrt(mean_squared_error(yt, yp)), 3),
            'R2':   round(r2_score(yt, yp), 4)
        })
    # Overall
    rows.append({
        'Horizon': 'AVERAGE',
        'MAE':  round(mean_absolute_error(Y_true.ravel(), Y_pred.ravel()), 3),
        'RMSE': round(np.sqrt(mean_squared_error(Y_true.ravel(), Y_pred.ravel())), 3),
        'R2':   round(np.mean([r2_score(Y_true[:, h], Y_pred[:, h]) for h in range(horizon)]), 4)
    })
    return pd.DataFrame(rows)


def get_models():
    return {
        'Linear Regression': LinearRegression(),
        'KNN': MultiOutputRegressor(KNeighborsRegressor(n_neighbors=5), n_jobs=-1),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
        'XGBoost': MultiOutputRegressor(
            XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE, verbosity=0),
            n_jobs=-1
        ),
        'CatBoost': CatBoostRegressor(
            loss_function='MultiRMSE', eval_metric='MultiRMSE',
            iterations=2000, learning_rate=0.03, depth=6,
            l2_leaf_reg=5, random_strength=1, subsample=0.8,
            bootstrap_type='Bernoulli', od_type='Iter', od_wait=100,
            verbose=200, random_seed=RANDOM_STATE
        ),
    }


def train_and_evaluate(X_train, Y_train, X_valid, Y_valid, X_test, Y_test,
                        cat_features=None, version_name=''):
    """Train tất cả models, trả về dict {model_name: df_metrics}."""
    models = get_models()
    cat_features = cat_features or []

    # Preprocessor cho non-CatBoost models
    preprocessor = build_preprocessor(X_train, categorical_cols=cat_features)
    X_train_proc = preprocessor.fit_transform(X_train)
    X_valid_proc = preprocessor.transform(X_valid)
    X_test_proc  = preprocessor.transform(X_test)

    results = {}
    for name, model in models.items():
        print(f'  [{version_name}] Training {name}...')
        if name == 'CatBoost':
            model.fit(
                X_train, Y_train,
                cat_features=cat_features if cat_features else None,
                eval_set=(X_valid, Y_valid),
                use_best_model=True
            )
            Y_pred = model.predict(X_test)
        else:
            model.fit(X_train_proc, Y_train)
            Y_pred = model.predict(X_test_proc)

        df_metrics = evaluate_mimo(Y_test.values, Y_pred)
        df_metrics.insert(0, 'Model', name)
        results[name] = {'metrics': df_metrics, 'model': model, 'Y_pred': Y_pred}

        avg_row = df_metrics[df_metrics['Horizon'] == 'AVERAGE'].iloc[0]
        print(f'         → MAE={avg_row["MAE"]:.3f}  RMSE={avg_row["RMSE"]:.3f}  R²={avg_row["R2"]:.4f}')

    return results

print('Utility functions loaded.')

Utility functions loaded.


---
## 3. PHIÊN BẢN 1 — Baseline (biến gốc từ dataset)

Features: toàn bộ cột meteo gốc từ `clean_data.csv` + `pm25_avg` hiện tại.  
Không tạo lag, rolling, hay encoding nào thêm.  
Mục đích: làm mốc tham chiếu để đánh giá giá trị của FE.

In [19]:
# --- Tạo targets ---
df_base = make_targets(df_raw, horizon=HORIZON)

# --- Drop NaN ---
# Bao gồm: hàng thiếu pm25_avg, hàng cuối thiếu target (shift âm),
# hàng thiếu meteo bất kỳ
df_base = df_base.dropna()
print(f'Shape sau dropna: {df_base.shape}')

# --- Định nghĩa X, Y ---
FEATURE_COLS_BASE = [c for c in df_raw.columns if c != 'pm25_avg'] + ['pm25_avg']
# pm25_avg hiện tại vẫn là feature (giá trị tại thời điểm t)
FEATURE_COLS_BASE = [c for c in FEATURE_COLS_BASE if c in df_base.columns]

X_base = df_base[FEATURE_COLS_BASE]
Y_base = df_base[TARGET_COLS]

print(f'Features ({len(FEATURE_COLS_BASE)}): {FEATURE_COLS_BASE}')
print(f'X shape: {X_base.shape} | Y shape: {Y_base.shape}')

# --- Split ---
train_b, valid_b, test_b = chrono_split(df_base)
X_train_b, Y_train_b = train_b[FEATURE_COLS_BASE], train_b[TARGET_COLS]
X_valid_b, Y_valid_b = valid_b[FEATURE_COLS_BASE], valid_b[TARGET_COLS]
X_test_b,  Y_test_b  = test_b[FEATURE_COLS_BASE],  test_b[TARGET_COLS]

print(f'\nTrain: {len(X_train_b):,}  Valid: {len(X_valid_b):,}  Test: {len(X_test_b):,}')
print(f'Train: {X_train_b.index.min()} → {X_train_b.index.max()}')
print(f'Test:  {X_test_b.index.min()}  → {X_test_b.index.max()}')

Shape sau dropna: (10631, 16)
Features (10): ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'wind_direction_10m', 'surface_pressure', 'boundary_layer_height', 'wind_u', 'wind_v', 'pm25_avg']
X shape: (10631, 10) | Y shape: (10631, 6)

Train: 7,441  Valid: 1,595  Test: 1,595
Train: 2024-11-19 10:00:00 → 2025-11-09 19:00:00
Test:  2026-01-15 07:00:00  → 2026-03-22 17:00:00


In [20]:
print('=== BASELINE TRAINING ===')
results_base = train_and_evaluate(
    X_train_b, Y_train_b,
    X_valid_b, Y_valid_b,
    X_test_b,  Y_test_b,
    cat_features=[],
    version_name='Baseline'
)

=== BASELINE TRAINING ===
  [Baseline] Training Linear Regression...
         → MAE=8.924  RMSE=12.722  R²=0.4075
  [Baseline] Training KNN...
         → MAE=9.304  RMSE=13.451  R²=0.3377
  [Baseline] Training Random Forest...
         → MAE=8.401  RMSE=12.434  R²=0.4340
  [Baseline] Training XGBoost...
         → MAE=8.729  RMSE=12.705  R²=0.4091
  [Baseline] Training CatBoost...
0:	learn: 39.4266109	test: 54.4850906	best: 54.4850906 (0)	total: 137ms	remaining: 4m 33s
200:	learn: 28.6103346	test: 38.5226299	best: 38.5226299 (200)	total: 1.51s	remaining: 13.6s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 38.38655831
bestIteration = 235

Shrink model to first 236 iterations.
         → MAE=8.192  RMSE=11.984  R²=0.4742


In [21]:
# Kết quả Baseline
df_summary_base = pd.concat(
    [v['metrics'][v['metrics']['Horizon'] == 'AVERAGE'] for v in results_base.values()],
    ignore_index=True
).sort_values('R2', ascending=False)

print('=== BASELINE — Overall (average across 6 horizons) ===')
display(df_summary_base)

# Per-horizon RMSE pivot
df_detail_base = pd.concat(
    [v['metrics'][v['metrics']['Horizon'] != 'AVERAGE'] for v in results_base.values()],
    ignore_index=True
)
print('\n=== BASELINE — RMSE per horizon ===')
display(df_detail_base.pivot_table(index='Model', columns='Horizon', values='RMSE', aggfunc='first'))

=== BASELINE — Overall (average across 6 horizons) ===


,Model,Horizon,MAE,RMSE,R2
4,CatBoost,AVERAGE,8.192,11.984,0.4742
2,Random Forest,AVERAGE,8.401,12.434,0.4340
3,XGBoost,AVERAGE,8.729,12.705,0.4091
0,Linear Regression,AVERAGE,8.924,12.722,0.4075
1,KNN,AVERAGE,9.304,13.451,0.3377



=== BASELINE — RMSE per horizon ===


Horizon,t+1h,t+2h,t+3h,t+4h,t+5h,t+6h
Model,,,,,,
CatBoost,8.027,10.673,12.001,12.810,13.423,13.968
KNN,9.268,11.722,13.431,14.297,15.013,15.875
Linear Regression,8.504,11.262,12.698,13.625,14.303,14.850
Random Forest,8.316,11.079,12.514,13.262,13.911,14.481
XGBoost,8.421,11.272,12.747,13.445,14.208,15.015


In [29]:
# @title Dashboard Ranking (Lưới 2x3) cho từng Lag (t+1 ... t+6)
from IPython.display import display, HTML

horizons = sorted([h for h in df_detail_all['Horizon'].unique() if h != 'AVERAGE'])

# Tạo danh sách các bảng (styler objects)
tables = []
for h in horizons:
    df_lag = (
        df_detail_all[df_detail_all['Horizon'] == h]
        .drop(columns=['Horizon'])
        .sort_values('RMSE', ascending=True)
        .reset_index(drop=True)
    )
    
    # Định dạng số và thêm tiêu đề bên trong bảng để dễ nhìn
    styler = df_lag.style.set_caption(f"<b>KẾT QUẢ DỰ BÁO: {h}</b>")\
                         .format({col: '{:.4f}' for col in df_lag.select_dtypes(include='number').columns})\
                         .set_table_styles([{'selector': 'caption', 'props': [('font-size', '14px'), ('color', '#1f3a8a')]}])
    tables.append(styler.to_html())

# Chia làm 2 hàng, mỗi hàng 3 bảng
for i in range(0, len(tables), 3):
    row_tables = tables[i:i+3]
    
    # Sử dụng Flexbox để dàn hàng ngang
    html_str = f"""
    <div style="display: flex; flex-direction: row; justify-content: space-between; margin-bottom: 30px;">
        <div style="flex: 1; margin-right: 10px;">{row_tables[0] if len(row_tables) > 0 else ""}</div>
        <div style="flex: 1; margin-right: 10px;">{row_tables[1] if len(row_tables) > 1 else ""}</div>
        <div style="flex: 1;">{row_tables[2] if len(row_tables) > 2 else ""}</div>
    </div>
    """
    display(HTML(html_str))


,Model,MAE,RMSE,R2
0,CatBoost,4.8490,8.0270,0.7645
1,Random Forest,4.9590,8.3160,0.7472
2,XGBoost,5.0500,8.4210,0.7408
3,Linear Regression,5.0430,8.5040,0.7356
4,KNN,6.0210,9.2680,0.6860
,Model,MAE,RMSE,R2
0,CatBoost,7.0650,10.6730,0.5829
1,Random Forest,7.2250,11.0790,0.5506
2,Linear Regression,7.5640,11.2620,0.5357
3,XGBoost,7.4100,11.2720,0.5348


,Model,MAE,RMSE,R2
0,CatBoost,9.1240,12.8100,0.3992
1,Random Forest,9.3730,13.2620,0.3560
2,XGBoost,9.6790,13.4450,0.3382
3,Linear Regression,9.9690,13.6250,0.3203
4,KNN,10.0870,14.2970,0.2516
,Model,MAE,RMSE,R2
0,CatBoost,9.6720,13.4230,0.3404
1,Random Forest,9.9690,13.9110,0.2915
2,XGBoost,10.3860,14.2080,0.2610
3,Linear Regression,10.7050,14.3030,0.2511


---
## 4. PHIÊN BẢN 2 — Feature Engineering

Tạo features theo kết luận từ EDA / ACF/PACF / Correlation analysis:

| Nhóm | Features | Ưu tiên |
|---|---|---|
| Autoregressive | lag_1h, lag_2h, lag_24h, lag_25h | Bắt buộc |
| Dự phòng | lag_3h – lag_6h | Dự phòng |
| Rolling | rolling_mean_3h, rolling_mean_24h, rolling_std_6h | Bắt buộc |
| Cyclical | hour_sin/cos, month_sin/cos | Bắt buộc |
| Weekly | day_of_week, is_weekend, lag_168h | Ít kỳ vọng |
| Meteo gốc | giữ nguyên từ dataset | — |

**Quan trọng:** rolling dùng `.shift(1).rolling(w)` để tránh data leakage  
(window không bao gồm timestep hiện tại).

In [30]:
df_fe = df_raw.copy()

# ── Autoregressive lags ──────────────────────────────────────────────────────
# Bắt buộc
df_fe['lag_1h']  = df_fe['pm25_avg'].shift(1)
df_fe['lag_2h']  = df_fe['pm25_avg'].shift(2)
df_fe['lag_24h'] = df_fe['pm25_avg'].shift(24)
df_fe['lag_25h'] = df_fe['pm25_avg'].shift(25)

# Dự phòng (PACF không còn rõ nhưng giữ để model tự chọn qua feature importance)
for h in range(3, 7):
    df_fe[f'lag_{h}h'] = df_fe['pm25_avg'].shift(h)

# Ít kỳ vọng — weekly
df_fe['lag_168h'] = df_fe['pm25_avg'].shift(168)

# ── Rolling statistics — shift(1) trước để tránh leakage ────────────────────
_shifted = df_fe['pm25_avg'].shift(1)   # không bao gồm timestep t

df_fe['rolling_mean_3h']  = _shifted.rolling(3,  min_periods=2).mean()
df_fe['rolling_mean_24h'] = _shifted.rolling(24, min_periods=12).mean()
df_fe['rolling_std_6h']   = _shifted.rolling(6,  min_periods=3).std()

# ── Cyclical time encoding ───────────────────────────────────────────────────
df_fe['hour_sin']  = np.sin(2 * np.pi * df_fe.index.hour / 24)
df_fe['hour_cos']  = np.cos(2 * np.pi * df_fe.index.hour / 24)
df_fe['month_sin'] = np.sin(2 * np.pi * df_fe.index.month / 12)
df_fe['month_cos'] = np.cos(2 * np.pi * df_fe.index.month / 12)

# ── Weekly context ───────────────────────────────────────────────────────────
df_fe['day_of_week'] = df_fe.index.dayofweek          # 0=Mon, 6=Sun
df_fe['is_weekend']  = (df_fe['day_of_week'] >= 5).astype(int)

# ── Targets ──────────────────────────────────────────────────────────────────
df_fe = make_targets(df_fe, horizon=HORIZON)

# ── Drop NaN ─────────────────────────────────────────────────────────────────
# Hàng đầu mỗi segment tự bị loại vì lag của chúng = NaN
# Hàng cuối bị loại vì target shift âm = NaN
n_before = len(df_fe)
df_fe = df_fe.dropna()
print(f'Rows trước dropna: {n_before:,} | Sau dropna: {len(df_fe):,} | Dropped: {n_before - len(df_fe):,}')

# ── Feature columns ───────────────────────────────────────────────────────────
# Loại pm25_avg ra khỏi features vì đã có lag_1h làm đại diện
# (pm25_avg tại t chính là lag_0h — nếu giữ, nó sẽ dominate và che mờ các lag khác)
FEATURE_COLS_FE = [c for c in df_fe.columns if c not in TARGET_COLS + ['pm25_avg']]
BINARY_COLS = ['is_weekend']

X_fe = df_fe[FEATURE_COLS_FE]
Y_fe = df_fe[TARGET_COLS]

print(f'\nFeatures ({len(FEATURE_COLS_FE)}):')
print(FEATURE_COLS_FE)
print(f'\nX shape: {X_fe.shape} | Y shape: {Y_fe.shape}')

Rows trước dropna: 11,726 | Sau dropna: 10,127 | Dropped: 1,599

Features (27):
['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'wind_direction_10m', 'surface_pressure', 'boundary_layer_height', 'wind_u', 'wind_v', 'lag_1h', 'lag_2h', 'lag_24h', 'lag_25h', 'lag_3h', 'lag_4h', 'lag_5h', 'lag_6h', 'lag_168h', 'rolling_mean_3h', 'rolling_mean_24h', 'rolling_std_6h', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_of_week', 'is_weekend']

X shape: (10127, 27) | Y shape: (10127, 6)


In [31]:
# --- Split ---
train_fe, valid_fe, test_fe = chrono_split(df_fe)
X_train_fe, Y_train_fe = train_fe[FEATURE_COLS_FE], train_fe[TARGET_COLS]
X_valid_fe, Y_valid_fe = valid_fe[FEATURE_COLS_FE], valid_fe[TARGET_COLS]
X_test_fe,  Y_test_fe  = test_fe[FEATURE_COLS_FE],  test_fe[TARGET_COLS]

print(f'Train: {len(X_train_fe):,}  Valid: {len(X_valid_fe):,}  Test: {len(X_test_fe):,}')
print(f'Train: {X_train_fe.index.min()} → {X_train_fe.index.max()}')
print(f'Test:  {X_test_fe.index.min()}  → {X_test_fe.index.max()}')

Train: 7,088  Valid: 1,519  Test: 1,520
Train: 2024-11-26 10:00:00 → 2025-11-16 02:00:00
Test:  2026-01-18 10:00:00  → 2026-03-22 17:00:00


In [32]:
print('=== FE VERSION TRAINING ===')
results_fe = train_and_evaluate(
    X_train_fe, Y_train_fe,
    X_valid_fe, Y_valid_fe,
    X_test_fe,  Y_test_fe,
    cat_features=[],   # không có cat feature trong FE version
    version_name='FE'
)

=== FE VERSION TRAINING ===
  [FE] Training Linear Regression...
         → MAE=9.495  RMSE=12.964  R²=0.3709
  [FE] Training KNN...
         → MAE=10.295  RMSE=14.662  R²=0.1954
  [FE] Training Random Forest...
         → MAE=8.988  RMSE=12.850  R²=0.3819
  [FE] Training XGBoost...
         → MAE=8.998  RMSE=13.011  R²=0.3664
  [FE] Training CatBoost...
0:	learn: 40.3028401	test: 53.6501367	best: 53.6501367 (0)	total: 21.7ms	remaining: 43.4s
200:	learn: 28.3592008	test: 39.1140379	best: 39.1098948 (199)	total: 2.84s	remaining: 25.4s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 39.01662601
bestIteration = 275

Shrink model to first 276 iterations.
         → MAE=8.753  RMSE=12.364  R²=0.4278


In [35]:
# @title Dashboard Ranking FE (Lưới 2x3) cho từng Lag (t+1 ... t+6)
from IPython.display import display, HTML

# 1. Tập hợp dữ liệu chi tiết từ kết quả FE (loại bỏ dòng AVERAGE)
df_detail_fe = pd.concat(
    [v['metrics'][v['metrics']['Horizon'] != 'AVERAGE'] for v in results_fe.values()],
    ignore_index=True
)

# 2. Lấy danh sách các mốc dự báo (Horizon) và sắp xếp theo thứ tự
horizons = sorted([h for h in df_detail_fe['Horizon'].unique() if h != 'AVERAGE'])

# 3. Tạo danh sách các bảng HTML
html_tables = []
for h in horizons:
    # Lọc data cho từng Horizon và sắp xếp theo RMSE tốt nhất
    df_lag = (
        df_detail_fe[df_detail_fe['Horizon'] == h]
        .drop(columns=['Horizon'])
        .sort_values('RMSE', ascending=True)
        .reset_index(drop=True)
    )
    
    # Định dạng bảng: Tiêu đề xanh đậm, số thập phân 4 chữ số
    styler = df_lag.style.set_caption(f"<b style='color: #1f3a8a;'>FE VERSION: {h}</b>")\
                         .format({col: '{:.4f}' for col in df_lag.select_dtypes(include='number').columns})\
                         .set_table_styles([
                             {'selector': 'caption', 'props': [('font-size', '14px'), ('padding', '10px')]},
                             {'selector': 'table', 'props': [('margin-left', 'auto'), ('margin-right', 'auto')]}
                         ])
    html_tables.append(styler.to_html())

# 4. Hiển thị theo lưới: 2 hàng, mỗi hàng 3 bảng
print("=== CHI TIẾT FE VERSION THEO TỪNG LAG (2 ROWS x 3 COLS) ===")
for i in range(0, len(html_tables), 3):
    current_row = html_tables[i:i+3]
    
    # Dàn hàng ngang bằng Flexbox
    html_grid = f"""
    <div style="display: flex; flex-direction: row; justify-content: space-around; align-items: flex-start; margin-bottom: 40px; gap: 20px;">
        <div style="flex: 1;">{current_row[0] if len(current_row) > 0 else ""}</div>
        <div style="flex: 1;">{current_row[1] if len(current_row) > 1 else ""}</div>
        <div style="flex: 1;">{current_row[2] if len(current_row) > 2 else ""}</div>
    </div>
    """
    display(HTML(html_grid))


=== CHI TIẾT FE VERSION THEO TỪNG LAG (2 ROWS x 3 COLS) ===


,Model,MAE,RMSE,R2
0,CatBoost,6.8160,10.4300,0.5922
1,Random Forest,7.1610,10.8460,0.5590
2,Linear Regression,7.2320,10.9070,0.5540
3,XGBoost,6.9940,10.9360,0.5517
4,KNN,8.8470,12.9420,0.3721
,Model,MAE,RMSE,R2
0,CatBoost,7.9810,11.5680,0.4987
1,Linear Regression,8.5170,12.0650,0.4548
2,XGBoost,8.1790,12.0660,0.4546
3,Random Forest,8.2670,12.1090,0.4508


,Model,MAE,RMSE,R2
0,CatBoost,9.2800,12.7980,0.3873
1,Random Forest,9.5330,13.3450,0.3338
2,Linear Regression,10.1640,13.4830,0.3198
3,XGBoost,9.6660,13.5180,0.3163
4,KNN,10.7180,14.9360,0.1654
,Model,MAE,RMSE,R2
0,CatBoost,9.6930,13.2490,0.3434
1,Random Forest,9.8980,13.7780,0.2899
2,Linear Regression,10.6410,13.9190,0.2753
3,XGBoost,9.9170,13.9980,0.2670


---
## 5. So sánh Baseline vs FE

In [36]:
# Bảng so sánh overall
df_compare = df_summary_base[['Model', 'MAE', 'RMSE', 'R2']].copy()
df_compare.columns = ['Model', 'MAE_base', 'RMSE_base', 'R2_base']

df_fe_sum = df_summary_fe[['Model', 'MAE', 'RMSE', 'R2']].copy()
df_fe_sum.columns = ['Model', 'MAE_fe', 'RMSE_fe', 'R2_fe']

df_compare = df_compare.merge(df_fe_sum, on='Model')
df_compare['RMSE_improvement'] = ((df_compare['RMSE_base'] - df_compare['RMSE_fe']) / df_compare['RMSE_base'] * 100).round(1)
df_compare['R2_improvement']   = (df_compare['R2_fe'] - df_compare['R2_base']).round(4)

print('=== BASELINE vs FE — Overall comparison ===')
display(df_compare.sort_values('RMSE_improvement', ascending=False))

=== BASELINE vs FE — Overall comparison ===


,Model,MAE_base,RMSE_base,R2_base,MAE_fe,RMSE_fe,R2_fe,RMSE_improvement,R2_improvement
3,Linear Regression,8.924,12.722,0.4075,9.495,12.964,0.3709,-1.9,-0.0366
2,XGBoost,8.729,12.705,0.4091,8.998,13.011,0.3664,-2.4,-0.0427
0,CatBoost,8.192,11.984,0.4742,8.753,12.364,0.4278,-3.2,-0.0464
1,Random Forest,8.401,12.434,0.4340,8.988,12.850,0.3819,-3.3,-0.0521
4,KNN,9.304,13.451,0.3377,10.295,14.662,0.1954,-9.0,-0.1423


---
## 6. Feature Importance — FE version (CatBoost)

CatBoost với `MultiRMSE` tính importance tổng hợp qua tất cả 6 horizons.

In [ ]:
cat_model_fe = results_fe['CatBoost']['model']

df_importance = pd.DataFrame({
    'feature':    X_train_fe.columns,
    'importance': cat_model_fe.get_feature_importance()
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('Top 20 features (CatBoost MultiRMSE — FE version):')
display(df_importance.head(20))

# Nhóm importance theo category
def categorize(feat):
    if feat.startswith('lag_'):     return 'AR Lag'
    if feat.startswith('rolling_'): return 'Rolling'
    if feat.endswith('_sin') or feat.endswith('_cos'): return 'Cyclical'
    if feat in ['day_of_week', 'is_weekend']: return 'Weekly'
    return 'Meteo'

df_importance['group'] = df_importance['feature'].apply(categorize)
df_group = df_importance.groupby('group')['importance'].sum().sort_values(ascending=False)
print('\nImportance by feature group (%):')
print((df_group / df_group.sum() * 100).round(1).to_string())

In [ ]:
# Barplot feature importance — top 20
group_colors = {
    'AR Lag': '#534AB7',
    'Rolling': '#1D9E75',
    'Cyclical': '#EF9F27',
    'Weekly': '#888780',
    'Meteo': '#D85A30'
}

top20 = df_importance.head(20)
colors = [group_colors.get(g, '#888') for g in top20['group']]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top 20 bar
axes[0].barh(top20['feature'][::-1], top20['importance'][::-1], color=colors[::-1])
axes[0].set_title('CatBoost Feature Importance — Top 20 (FE version)')
axes[0].set_xlabel('Importance')

# Add legend
from matplotlib.patches import Patch
legend_handles = [Patch(color=c, label=g) for g, c in group_colors.items()]
axes[0].legend(handles=legend_handles, fontsize=8, loc='lower right')

# Group pie
axes[1].pie(
    df_group.values,
    labels=df_group.index,
    colors=[group_colors.get(g, '#888') for g in df_group.index],
    autopct='%1.1f%%', startangle=140
)
axes[1].set_title('Importance by Feature Group')

plt.tight_layout()
plt.savefig('feature_importance_fe.png', dpi=120, bbox_inches='tight')
plt.show()